# cluster01 bike_change 2차 피처 조합 비교

## 용어 설명

- `subset`(축소 피처 조합): 전체 후보 피처 중 일부만 골라 만든 비교안
- `commute + transit`: 출퇴근 시간대 + 교통 접근성 중심 조합
- `short trend`: 짧은 구간의 과거값과 이동 통계

목표:
- `cluster01`에서 어떤 축소 피처 조합이 `bike_change`에 더 유리한지 확인한다.


In [1]:
from pathlib import Path

import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path('/Users/cheng80/Desktop/ddri_work')
WORK_DIR = ROOT / 'works/07_prediction_bike_change'
TRAIN_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_train_2023_2024.csv'
TEST_PATH = WORK_DIR / 'output/data/ddri_prediction_bike_change_test_2025.csv'
OUTPUT_DIR = WORK_DIR / 'output/data'

TARGET_CLUSTER = 1
TARGET_STATION_GROUP = '아침 도착 업무 집중형'
RANDOM_STATE = 42


In [2]:
FULL_FEATURES = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h', 'rolling_std_24h',
    'rolling_mean_168h', 'rolling_std_168h', 'rolling_mean_6h', 'is_weekend', 'is_night_hour',
    'is_commute_hour', 'hour_sin', 'is_rainy', 'hour_cos', 'commute_morning_flag',
    'commute_evening_flag', 'subway_distance_m', 'distance_naturepark_m', 'restaurant_count_300m',
    'convenience_store_count_300m', 'bus_stop_count_300m', 'cafe_count_300m',
    'elevation_diff_nearest_subway_m', 'nearest_park_area_sqm'
]

SUBSET_A = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'is_commute_hour', 'commute_morning_flag', 'commute_evening_flag',
    'subway_distance_m', 'bus_stop_count_300m'
]

SUBSET_B = SUBSET_A + [
    'lag_24h', 'lag_168h', 'rolling_mean_6h'
]

SUBSET_C = [
    'station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday',
    'temperature', 'precipitation', 'is_commute_hour', 'commute_morning_flag', 'commute_evening_flag',
    'lag_24h', 'lag_168h', 'rolling_mean_6h', 'hour_sin', 'hour_cos',
    'subway_distance_m', 'bus_stop_count_300m'
]

FEATURE_SETS = {
    'full_36': FULL_FEATURES,
    'subset_a_commute_transit': SUBSET_A,
    'subset_b_commute_transit_short_trend': SUBSET_B,
    'subset_c_current_compact': SUBSET_C,
}

BASE_CATEGORICAL = ['station_id', 'cluster', 'mapped_dong_code', 'hour', 'weekday', 'month', 'holiday']


In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_group = train_raw.loc[train_raw['cluster'] == TARGET_CLUSTER].copy()
test_group = test_raw.loc[test_raw['cluster'] == TARGET_CLUSTER].copy()

train_2023 = train_group.loc[train_group['date'] < '2024-01-01'].copy()
valid_2024 = train_group.loc[train_group['date'] >= '2024-01-01'].copy()
test_2025 = test_group.copy()
full_train = train_group.copy()


In [4]:
def evaluate_predictions(model_name: str, split_name: str, y_true: pd.Series, y_pred) -> dict:
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sign_accuracy = ((pd.Series(y_pred) >= 0) == (y_true.reset_index(drop=True) >= 0)).mean()
    return {
        'model': model_name,
        'split': split_name,
        'rmse': round(float(rmse), 4),
        'mae': round(float(mae), 4),
        'r2': round(float(r2), 4),
        'sign_accuracy': round(float(sign_accuracy), 4),
    }

def prepare_xy(df: pd.DataFrame, feature_cols: list[str]):
    X = df[feature_cols].copy()
    for col in [c for c in feature_cols if c in BASE_CATEGORICAL]:
        X[col] = X[col].astype('category')
    y = df['bike_change'].copy()
    return X, y

def build_lgbm_model():
    return LGBMRegressor(
        objective='regression',
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
    )


In [5]:
results = []

for feature_set_name, feature_cols in FEATURE_SETS.items():
    X_train, y_train = prepare_xy(train_2023, feature_cols)
    X_valid, y_valid = prepare_xy(valid_2024, feature_cols)
    X_full, y_full = prepare_xy(full_train, feature_cols)
    X_test, y_test = prepare_xy(test_2025, feature_cols)

    cat_cols = [c for c in feature_cols if c in BASE_CATEGORICAL]
    model = build_lgbm_model()
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_2023', y_train, model.predict(X_train)))
    results.append(evaluate_predictions(feature_set_name, 'validation_2024', y_valid, model.predict(X_valid)))

    model = build_lgbm_model()
    model.fit(X_full, y_full, categorical_feature=cat_cols)
    results.append(evaluate_predictions(feature_set_name, 'train_full_refit', y_full, model.predict(X_full)))
    results.append(evaluate_predictions(feature_set_name, 'test_2025_refit', y_test, model.predict(X_test)))

results_df = pd.DataFrame(results)
results_df['cluster'] = TARGET_CLUSTER
results_df['station_group'] = TARGET_STATION_GROUP
results_df['feature_count'] = results_df['model'].map({k: len(v) for k, v in FEATURE_SETS.items()})

metrics_path = OUTPUT_DIR / 'ddri_bike_change_cluster01_second_round_metrics.csv'
results_df.to_csv(metrics_path, index=False, encoding='utf-8-sig')
print('saved:', metrics_path)
results_df


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005020 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1860
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 34
[LightGBM] [Info] Start training from score 0.000078


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1905
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 34
[LightGBM] [Info] Start training from score 0.000019


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.034073 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 71
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 11
[LightGBM] [Info] Start training from score 0.000078


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035847 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 71
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 11
[LightGBM] [Info] Start training from score 0.000019


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000890 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 195
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 14
[LightGBM] [Info] Start training from score 0.000078


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002269 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 208
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 14
[LightGBM] [Info] Start training from score 0.000019


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002158 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 581
[LightGBM] [Info] Number of data points in the train set: 25776, number of used features: 18
[LightGBM] [Info] Start training from score 0.000078


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010438 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 614
[LightGBM] [Info] Number of data points in the train set: 52128, number of used features: 18
[LightGBM] [Info] Start training from score 0.000019


saved: /Users/cheng80/Desktop/ddri_work/works/07_prediction_bike_change/output/data/ddri_bike_change_cluster01_second_round_metrics.csv


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,train_2023,1.0775,0.7216,0.7898,0.8672,1,아침 도착 업무 집중형,35
1,full_36,validation_2024,1.5596,0.9205,0.5759,0.8460,1,아침 도착 업무 집중형,35
2,full_36,train_full_refit,1.2026,0.7832,0.7431,0.8709,1,아침 도착 업무 집중형,35
3,full_36,test_2025_refit,1.5716,0.9531,0.6243,0.8639,1,아침 도착 업무 집중형,35
4,subset_a_commute_transit,train_2023,1.8872,1.1460,0.3550,0.5727,1,아침 도착 업무 집중형,12
5,subset_a_commute_transit,validation_2024,2.0347,1.1914,0.2781,0.5364,1,아침 도착 업무 집중형,12
6,subset_a_commute_transit,train_full_refit,1.9209,1.1437,0.3446,0.5543,1,아침 도착 업무 집중형,12
7,subset_a_commute_transit,test_2025_refit,2.1227,1.2667,0.3146,0.5335,1,아침 도착 업무 집중형,12
8,subset_b_commute_transit_short_trend,train_2023,1.5896,1.0179,0.5424,0.6700,1,아침 도착 업무 집중형,15
9,subset_b_commute_transit_short_trend,validation_2024,1.9208,1.1421,0.3567,0.6411,1,아침 도착 업무 집중형,15


In [6]:
results_df[results_df['split'] == 'test_2025_refit'].sort_values('rmse').reset_index(drop=True)


,model,split,rmse,mae,r2,sign_accuracy,cluster,station_group,feature_count
0,full_36,test_2025_refit,1.5716,0.9531,0.6243,0.8639,1,아침 도착 업무 집중형,35
1,subset_c_current_compact,test_2025_refit,1.9536,1.1767,0.4194,0.6455,1,아침 도착 업무 집중형,19
2,subset_b_commute_transit_short_trend,test_2025_refit,1.9789,1.1927,0.4043,0.6513,1,아침 도착 업무 집중형,15
3,subset_a_commute_transit,test_2025_refit,2.1227,1.2667,0.3146,0.5335,1,아침 도착 업무 집중형,12
